# End-to-End EBSD Scan Band Width Workflow

This tutorial describes the production scan workflow for EDAX/TSL `.oh5`/`.h5` inputs and HKL/Oxford `.ctf` plus pattern-folder inputs. The outputs are CSV summaries, modified HDF5/OH5 datasets, and ANG exports.


## Workflow Summary

1. Validate one pattern in debug mode.
2. Configure scan source and phase.
3. Run the automator.
4. Inspect CSV, HDF5/OH5, and ANG outputs.
5. Compare outputs across TSL and CTF sources if both are available for the same sample.


In [ ]:
from pathlib import Path

repo = Path.cwd()
tsl_config = repo / "bandDetectorOptionsHcp.yml"
ctf_config = repo / "configs" / "ctf_ni_band_width_example.yml"
print("TSL config:", tsl_config)
print("CTF example config:", ctf_config)


## TSL OH5/H5 Run

For EDAX/TSL data, set `h5_file_path` in the YAML. The automator copies the source file, runs indexing, measures band widths, writes HDF5 datasets, writes an `.oh5` copy, and exports a companion `.ang` file using PRIAS columns.


In [ ]:
import subprocess, sys

cmd = [sys.executable, "KikuchiBandWidthAutomator.py", "--config", str(tsl_config)]
print(" ".join(cmd))
# subprocess.run(cmd, check=True)


## HKL/Oxford CTF Run

For CTF data, set `ctf_file_path`, `pattern_folder`, `phase_list`, `hkl_list`, and `ctf_detector`. If no `band_annotation_json_path` is supplied, the automator simulates line positions directly from CTF Euler angles.


In [ ]:
ctf_yaml = """
ctf_file_path: path/to/Ni_scan.ctf
pattern_folder: path/to/Ni_patterns
desired_hkl: "111"
desired_hkl_ref_width: 1.0
elastic_modulus: 200000000000.0
rectWidth: 20
min_psnr: 1.05
hkl_list:
  - [1, 1, 1]
phase_list:
  name: Ni
  space_group: 225
  lattice: [3.5236, 3.5236, 3.5236, 90, 90, 90]
  atoms:
    - element: Ni
      position: [0, 0, 0]
ctf_detector:
  convention: oxford
  pc: [0.5, 0.5, 0.5]
  sample_tilt: 70.0
  tilt: 0.0
  azimuthal: 0.0
ctf_euler_direction: lab2crystal
"""
print(ctf_yaml)


## Inspect HDF5/OH5 Outputs

The modified HDF5/OH5 stores scalar maps and band-profile metadata under `/scan/EBSD/Data`. Important datasets include `Band_Width`, `psnr`, `band_intensity_ratio`, `band_intensity_diff_norm`, `band_profile`, `central_line`, `band_start_idx`, `band_end_idx`, and `band_valid`.


In [ ]:
import h5py

modified_h5 = Path("path/to/scan_modified.h5")
if modified_h5.exists():
    with h5py.File(modified_h5, "r") as handle:
        scan_name = next(name for name in handle if name not in {"Manufacturer", "Version"})
        data = handle[f"/{scan_name}/EBSD/Data"]
        print("Scan:", scan_name)
        print("Datasets:", sorted(data.keys()))
else:
    print("Set modified_h5 to a completed output file to inspect datasets.")


## Cross-Format Checks

When both TSL and CTF are available for the same Ni sample, compare:

- grid dimensions and row-major pixel order;
- Euler angle convention and detector pattern center;
- scalar maps (`IQ`, `CI`/`MAD`, `Phase`);
- final `Band_Width`, `psnr`, and valid-pixel masks;
- ANG row count and PRIAS column mapping.

Large disagreement usually indicates PC convention mismatch, wrong Euler direction, pattern-folder ordering, or a mismatched phase/HKL selection.


## Acceptance Criteria

A scan run is production-ready when:

- debug single-pattern output is visually correct;
- full scan completes without interactive prompts in normal mode;
- `.csv`, `.h5`, `.oh5`, and `.ang` outputs are written;
- HDF5 dataset lengths match scan pixel count;
- ANG row count matches `nRows * nColumns`;
- warnings are reviewed and either corrected or documented.
